# Demo 1 : Dataset-to-Genie Walkthrough

**Module 5A - Databricks SQL + Genie (Topics 1.1-1.6)**

This notebook walks through the Databricks AI/BI family - from designing dashboard datasets to using Genie for ad-hoc data exploration. We cover dashboard building blocks, dataset sources, Genie Code for AI-assisted authoring, and the decision framework for choosing dashboards vs. Genie.


In [0]:
%sql
-- SETUP: Create catalog, schema, and sample data for dashboard + Genie demos
-- We create a retail sales dataset that's realistic for dashboards and Genie queries.

CREATE CATALOG IF NOT EXISTS module5a_demo1;
CREATE SCHEMA IF NOT EXISTS module5a_demo1.sales;

-- Customers table
CREATE OR REPLACE TABLE module5a_demo1.sales.customers (
  customer_id    STRING NOT NULL,
  customer_name  STRING NOT NULL,
  customer_tier  STRING,
  region         STRING,
  signup_date    DATE
);

INSERT INTO module5a_demo1.sales.customers VALUES
('CUST-001', 'Acme Corp',        'enterprise', 'North America', '2024-01-15'),
('CUST-002', 'Globex Inc',       'premium',    'Europe',        '2024-03-20'),
('CUST-003', 'Initech LLC',      'standard',   'North America', '2024-06-10'),
('CUST-004', 'Umbrella SA',      'enterprise', 'Asia Pacific',  '2024-02-05'),
('CUST-005', 'Stark Industries', 'premium',    'Europe',        '2024-08-18'),
('CUST-006', 'Wayne Tech',       'standard',   'North America', '2024-04-22'),
('CUST-007', 'Soylent Co',       'premium',    'Asia Pacific',  '2024-05-14'),
('CUST-008', 'Cyberdyne Sys',    'enterprise', 'Europe',        '2024-07-01');

-- Products table
CREATE OR REPLACE TABLE module5a_demo1.sales.products (
  product_id     STRING NOT NULL,
  product_name   STRING NOT NULL,
  category       STRING,
  unit_price     DECIMAL(10,2)
);

INSERT INTO module5a_demo1.sales.products VALUES
('PROD-001', 'Databricks Pro License',   'Software',  499.00),
('PROD-002', 'Databricks Enterprise',    'Software', 1499.00),
('PROD-003', 'Vector Search Add-on',    'Add-on',    299.00),
('PROD-004', 'AI Gateway Starter',      'Add-on',    199.00),
('PROD-005', 'Pro Support Package',     'Support',   999.00),
('PROD-006', 'Premium Training Voucher', 'Training',  499.00);

-- Orders table (fact table — the core of our dashboard + Genie queries)
CREATE OR REPLACE TABLE module5a_demo1.sales.orders (
  order_id       STRING NOT NULL,
  customer_id    STRING NOT NULL,
  product_id     STRING NOT NULL,
  order_date     DATE,
  quantity       INT,
  unit_price     DECIMAL(10,2),
  total_amount   DECIMAL(12,2),
  status         STRING,
  channel        STRING
);

INSERT INTO module5a_demo1.sales.orders VALUES
('ORD-001', 'CUST-001', 'PROD-002', '2025-01-15', 3, 1499.00, 4497.00, 'closed_won', 'direct'),
('ORD-002', 'CUST-002', 'PROD-001', '2025-01-28', 5,  499.00, 2495.00, 'closed_won', 'partner'),
('ORD-003', 'CUST-003', 'PROD-003', '2025-02-10', 2,  299.00,  598.00, 'closed_won', 'direct'),
('ORD-004', 'CUST-004', 'PROD-002', '2025-02-22', 2, 1499.00, 2998.00, 'closed_won', 'direct'),
('ORD-005', 'CUST-005', 'PROD-005', '2025-03-05', 1,  999.00,  999.00, 'closed_won', 'partner'),
('ORD-006', 'CUST-001', 'PROD-001', '2025-04-12', 4,  499.00, 1996.00, 'closed_won', 'direct'),
('ORD-007', 'CUST-006', 'PROD-004', '2025-04-25', 3,  199.00,  597.00, 'closed_won', 'direct'),
('ORD-008', 'CUST-007', 'PROD-002', '2025-05-18', 2, 1499.00, 2998.00, 'closed_won', 'partner'),
('ORD-009', 'CUST-008', 'PROD-003', '2025-06-08', 5,  299.00, 1495.00, 'closed_won', 'direct'),
('ORD-010', 'CUST-002', 'PROD-006', '2025-06-20', 3,  499.00, 1497.00, 'closed_won', 'direct'),
('ORD-011', 'CUST-003', 'PROD-001', '2025-07-15', 2,  499.00,  998.00, 'closed_won', 'direct'),
('ORD-012', 'CUST-004', 'PROD-005', '2025-08-01', 1,  999.00,  999.00, 'closed_won', 'partner'),
('ORD-013', 'CUST-005', 'PROD-002', '2025-08-22', 3, 1499.00, 4497.00, 'closed_won', 'direct'),
('ORD-014', 'CUST-001', 'PROD-004', '2025-09-10', 4,  199.00,  796.00, 'closed_won', 'partner'),
('ORD-015', 'CUST-006', 'PROD-002', '2025-09-25', 1, 1499.00, 1499.00, 'pending',    'direct');

SELECT * FROM module5a_demo1.sales.orders ORDER BY order_date;

order_id,customer_id,product_id,order_date,quantity,unit_price,total_amount,status,channel
ORD-001,CUST-001,PROD-002,2025-01-15,3,1499.00,4497.00,closed_won,direct
ORD-002,CUST-002,PROD-001,2025-01-28,5,499.00,2495.00,closed_won,partner
ORD-003,CUST-003,PROD-003,2025-02-10,2,299.00,598.00,closed_won,direct
ORD-004,CUST-004,PROD-002,2025-02-22,2,1499.00,2998.00,closed_won,direct
ORD-005,CUST-005,PROD-005,2025-03-05,1,999.00,999.00,closed_won,partner
ORD-006,CUST-001,PROD-001,2025-04-12,4,499.00,1996.00,closed_won,direct
ORD-007,CUST-006,PROD-004,2025-04-25,3,199.00,597.00,closed_won,direct
ORD-008,CUST-007,PROD-002,2025-05-18,2,1499.00,2998.00,closed_won,partner
ORD-009,CUST-008,PROD-003,2025-06-08,5,299.00,1495.00,closed_won,direct
ORD-010,CUST-002,PROD-006,2025-06-20,3,499.00,1497.00,closed_won,direct


## 1.1 : The Databricks AI/BI Family

### Concepts
The Databricks AI/BI family provides four complementary ways to access and analyze data:

| Component | What it is | Best for |
|---|---|---|
| **Dashboards** | Visual, structured reports on UC data | Known, repeated questions for many users |
| **Genie Agent** | NL interface to governed data | Ad-hoc, exploratory questions |
| **Genie Code** | AI-assisted authoring in notebooks/dashboards | Generating SQL, visualizations, full dashboards from prompts |
| **Genie One** | Unified search across dashboards + Genie | Finding the right answer regardless of source |

**How they fit together**:
* **Dashboards** answer the questions you *know you have* (revenue by quarter, top customers).
* **Genie** answers the questions you *didn't know you'd have* ("which enterprise customers in Europe haven't ordered in Q3?").
* **Genie Code** helps you *build* dashboards and queries faster.
* **Genie One** ties it all together with a single search bar.

> All components are governed by Unity Catalog — the same permissions, lineage, and audit logs apply everywhere.

In [0]:
%sql
-- 1.1 Demo: The same data, accessed four ways
-- This single query shows the structured pattern that Dashboards use.
-- Genie would generate similar SQL from natural language.
-- Genie Code would write this SQL from a prompt like 'show monthly revenue'.
-- Genie One would find this result whether it lives in a dashboard or Genie.

SELECT 
  date_format(order_date, 'yyyy-MM') AS month,
  count(*) AS order_count,
  sum(total_amount) AS revenue
FROM module5a_demo1.sales.orders
WHERE status = 'closed_won'
GROUP BY 1
ORDER BY 1;

month,order_count,revenue
2025-01,2,6992.00
2025-02,2,3596.00
2025-03,1,999.00
2025-04,2,2593.00
2025-05,1,2998.00
2025-06,2,2992.00
2025-07,1,998.00
2025-08,2,5496.00
2025-09,1,796.00


## 1.2 : AI/BI Dashboard Building Blocks

### Concepts
A Databricks AI/BI Dashboard is built from five components:

1. **Datasets**: SQL queries that define what data the dashboard shows. Each dataset is a SELECT statement against UC tables or views.
2. **Canvas**: The layout surface where you place visualizations, text, and filters.
3. **Visualizations**: Bar charts, line charts, tables, KPIs, counters — each bound to a dataset.
4. **Filters**: Interactive controls (dropdowns, date ranges) that let users narrow the data.
5. **Pages**: A dashboard can have multiple pages, each with its own canvas and visualizations.

**Design principle**: Build one well-designed dataset per page, then create multiple visualizations from it. This avoids redundant queries and keeps the dashboard fast.

> A good dataset pre-joins and pre-aggregates so visualizations don't recompute everything on each load.

In [0]:
%sql
-- 1.2 Demo: Designing a dataset for a dashboard
-- A well-designed dataset pre-joins orders + customers + products
-- so every visualization on the dashboard uses one clean source.

CREATE OR REPLACE TABLE module5a_demo1.sales.dashboard_dataset AS
SELECT 
  o.order_id,
  o.order_date,
  date_format(o.order_date, 'yyyy-MM') AS order_month,
  concat(year(o.order_date), '-Q', quarter(o.order_date)) AS order_quarter,
  o.customer_id,
  c.customer_name,
  c.customer_tier,
  c.region,
  o.product_id,
  p.product_name,
  p.category AS product_category,
  o.quantity,
  o.unit_price,
  o.total_amount,
  o.status,
  o.channel
FROM module5a_demo1.sales.orders o
JOIN module5a_demo1.sales.customers c ON o.customer_id = c.customer_id
JOIN module5a_demo1.sales.products p ON o.product_id = p.product_id;

-- This single dataset powers all dashboard visualizations:
-- KPIs, trends, breakdowns by region/category/tier
SELECT * FROM module5a_demo1.sales.dashboard_dataset ORDER BY order_date;

order_id,order_date,order_month,order_quarter,customer_id,customer_name,customer_tier,region,product_id,product_name,product_category,quantity,unit_price,total_amount,status,channel
ORD-001,2025-01-15,2025-01,2025-Q1,CUST-001,Acme Corp,enterprise,North America,PROD-002,Databricks Enterprise,Software,3,1499.00,4497.00,closed_won,direct
ORD-002,2025-01-28,2025-01,2025-Q1,CUST-002,Globex Inc,premium,Europe,PROD-001,Databricks Pro License,Software,5,499.00,2495.00,closed_won,partner
ORD-003,2025-02-10,2025-02,2025-Q1,CUST-003,Initech LLC,standard,North America,PROD-003,Vector Search Add-on,Add-on,2,299.00,598.00,closed_won,direct
ORD-004,2025-02-22,2025-02,2025-Q1,CUST-004,Umbrella SA,enterprise,Asia Pacific,PROD-002,Databricks Enterprise,Software,2,1499.00,2998.00,closed_won,direct
ORD-005,2025-03-05,2025-03,2025-Q1,CUST-005,Stark Industries,premium,Europe,PROD-005,Pro Support Package,Support,1,999.00,999.00,closed_won,partner
ORD-006,2025-04-12,2025-04,2025-Q2,CUST-001,Acme Corp,enterprise,North America,PROD-001,Databricks Pro License,Software,4,499.00,1996.00,closed_won,direct
ORD-007,2025-04-25,2025-04,2025-Q2,CUST-006,Wayne Tech,standard,North America,PROD-004,AI Gateway Starter,Add-on,3,199.00,597.00,closed_won,direct
ORD-008,2025-05-18,2025-05,2025-Q2,CUST-007,Soylent Co,premium,Asia Pacific,PROD-002,Databricks Enterprise,Software,2,1499.00,2998.00,closed_won,partner
ORD-009,2025-06-08,2025-06,2025-Q2,CUST-008,Cyberdyne Sys,enterprise,Europe,PROD-003,Vector Search Add-on,Add-on,5,299.00,1495.00,closed_won,direct
ORD-010,2025-06-20,2025-06,2025-Q2,CUST-002,Globex Inc,premium,Europe,PROD-006,Premium Training Voucher,Training,3,499.00,1497.00,closed_won,direct


## 1.3 : Dashboard Dataset Sources

### Concepts
A dashboard dataset can pull from three sources:

| Source | What it is | When to use |
|---|---|---|
| **UC table** | Any Unity Catalog table or view queried directly | Simple datasets, single-table queries |
| **Reusable aggregation** | A pre-computed aggregation stored as a UC table | Shared KPIs used across multiple dashboards |
| **Dashboard-local view** | A query defined within the dashboard SQL editor | One-off calculations specific to one dashboard |

**Key distinction**:
* UC tables and reusable aggregations are **governed objects** — they have their own permissions, lineage, and audit trail.
* Dashboard-local views are **embedded in the dashboard** — they don't exist as separate UC objects.

> Prefer reusable UC aggregations for KPIs that multiple dashboards need. Use dashboard-local views for one-off calculations.

In [0]:
%sql
-- 1.3 Demo: Three dashboard dataset sources in action

-- Source 1: Direct UC table query (simplest)
SELECT 
  order_quarter,
  sum(total_amount) AS revenue,
  count(*) AS orders
FROM module5a_demo1.sales.dashboard_dataset
WHERE status = 'closed_won'
GROUP BY 1
ORDER BY 1;

-- Source 2: Reusable UC aggregation (pre-computed KPIs for multiple dashboards)
CREATE OR REPLACE TABLE module5a_demo1.sales.quarterly_revenue AS
SELECT 
  order_quarter,
  region,
  customer_tier,
  sum(total_amount) AS total_revenue,
  count(*) AS order_count,
  avg(total_amount) AS avg_order_value
FROM module5a_demo1.sales.dashboard_dataset
WHERE status = 'closed_won'
GROUP BY 1, 2, 3;

SELECT * FROM module5a_demo1.sales.quarterly_revenue ORDER BY order_quarter;

-- Source 3: Dashboard-local view (inline in the dashboard SQL editor)
-- This query would be defined directly in the dashboard, not as a UC object
SELECT 
  order_quarter,
  product_category,
  sum(total_amount) AS revenue
FROM module5a_demo1.sales.dashboard_dataset
WHERE status = 'closed_won'
GROUP BY 1, 2
ORDER BY 1, 2;

order_quarter,product_category,revenue
2025-Q1,Add-on,598.00
2025-Q1,Software,9990.00
2025-Q1,Support,999.00
2025-Q2,Add-on,2092.00
2025-Q2,Software,4994.00
2025-Q2,Training,1497.00
2025-Q3,Add-on,796.00
2025-Q3,Software,5495.00
2025-Q3,Support,999.00


## 1.4 : Genie Code

### Concepts
Genie Code is Databricks' AI-assisted authoring tool. It lives inside notebooks and dashboard editors and generates:

* **SQL from natural language**: Type "show revenue by quarter for each region" and get a ready-to-run SELECT.
* **Auto-configured visualizations**: Genie Code picks the chart type, axes, and grouping based on the data.
* **Full dashboard planning**: Describe what you want ("build a sales performance dashboard") and Genie Code plans pages, datasets, and widgets.
* **Inline quick-fixes**: Ask Genie Code to fix or modify existing SQL ("filter to closed_won only", "add a YoY comparison").

**How it differs from Genie Agent**:
* Genie Code helps you *build* artifacts (SQL, dashboards). The output is code you can edit and save.
* Genie Agent *answers questions* at runtime. The output is a natural language answer backed by auto-generated SQL.

> Genie Code is available in the notebook editor and the AI/BI Dashboard editor. Look for the AI button in the toolbar.

In [0]:
# 1.4 Demo: Genie Code — AI-assisted dashboard authoring
# Genie Code generates SQL, datasets, and visualizations from natural language.
# Here we simulate what Genie Code would produce from a prompt.

print("=== Genie Code: Natural Language to SQL + Visualization ===")
print()
print("User prompt: 'Show me revenue by quarter for each region'")
print()
print("Genie Code generates:")
print("  1. A SQL dataset query")
print("  2. Auto-detected visualization type (grouped bar chart)")
print("  3. Suggested filters (region, quarter)")
print()
print("--- Generated SQL ---")
print("""
SELECT
  order_quarter,
  region,
  SUM(total_amount) AS revenue
FROM module5a_demo1.sales.dashboard_dataset
WHERE status = 'closed_won'
GROUP BY order_quarter, region
ORDER BY order_quarter, region
""")
print("--- Auto-configured visualization ---")
print("  Type: Grouped bar chart")
print("  X-axis: order_quarter")
print("  Y-axis: revenue")
print("  Group by: region")
print()

print("=== Genie Code: Full Dashboard Planning ===")
print("User prompt: 'Build a sales performance dashboard'")
print()
print("Genie Code plans the full dashboard:")
print("  Page 1: KPIs (total revenue, order count, avg order value)")
print("  Page 2: Trends (revenue by quarter, orders by month)")
print("  Page 3: Breakdown (revenue by region, by product category)")
print("  Each page: auto-generated dataset + suggested visualizations")
print()

print("=== Genie Code: Inline Quick-Fix ===")
print("User prompt: 'Add a year-over-year comparison to the revenue chart'")
print()
print("Genie Code modifies the existing query:")
print("  - Adds a previous-year revenue column using LAG()")
print("  - Updates the visualization to show both current and prior year")
print("  - Suggests a line chart with two series")

=== Genie Code: Natural Language to SQL + Visualization ===

User prompt: 'Show me revenue by quarter for each region'

Genie Code generates:
  1. A SQL dataset query
  2. Auto-detected visualization type (grouped bar chart)
  3. Suggested filters (region, quarter)

--- Generated SQL ---

SELECT
  order_quarter,
  region,
  SUM(total_amount) AS revenue
FROM module5a_demo1.sales.dashboard_dataset
WHERE status = 'closed_won'
GROUP BY order_quarter, region
ORDER BY order_quarter, region

--- Auto-configured visualization ---
  Type: Grouped bar chart
  X-axis: order_quarter
  Y-axis: revenue
  Group by: region

=== Genie Code: Full Dashboard Planning ===
User prompt: 'Build a sales performance dashboard'

Genie Code plans the full dashboard:
  Page 1: KPIs (total revenue, order count, avg order value)
  Page 2: Trends (revenue by quarter, orders by month)
  Page 3: Breakdown (revenue by region, by product category)
  Each page: auto-generated dataset + suggested visualizations

=== Genie 

## 1.5 : Dashboard vs. Genie Agent

### Concepts
Dashboards and Genie Agent solve different problems with the same data:

| Aspect | Dashboard | Genie Agent |
|---|---|---|
| **Question type** | Known, repeated | Unknown, ad-hoc |
| **Output** | Visual (charts, KPIs, tables) | Natural language + table results |
| **Setup** | Build datasets + visualizations once | Point at tables, add instructions |
| **User** | Many users, broad audience | Individual exploring data |
| **Speed** | Instant (pre-built) | A few seconds (generates SQL on the fly) |
| **Flexibility** | Low (fixed visualizations) | High (any question within scope) |
| **Governance** | UC permissions on underlying tables | UC permissions + Genie-specific instructions |

**Decision framework**:
* **Use a Dashboard** when: the question is asked repeatedly, many users need the answer, and a visual format is best.
* **Use Genie** when: the question is new or exploratory, one person needs the answer right now, and flexibility matters more than polish.
* **Use both**: dashboards for the 80% known questions, Genie for the 20% ad-hoc questions on the same data.

In [0]:
%sql
-- 1.5 Demo: Dashboard vs. Genie Agent — same data, different access pattern

-- DASHBOARD question (known, repeated): "Revenue by quarter"
-- Pre-built, visual, shared with many users
SELECT 
  order_quarter,
  sum(total_amount) AS revenue,
  count(*) AS orders
FROM module5a_demo1.sales.dashboard_dataset
WHERE status = 'closed_won'
GROUP BY 1
ORDER BY 1;

-- GENIE question (ad-hoc, one-time): 
-- "Which enterprise customers in Europe haven't placed an order in Q3?"
-- Genie generates this SQL from natural language
SELECT c.customer_name, c.region, max(o.order_date) AS last_order
FROM module5a_demo1.sales.customers c
LEFT JOIN module5a_demo1.sales.orders o 
  ON c.customer_id = o.customer_id AND o.status = 'closed_won'
WHERE c.customer_tier = 'enterprise' AND c.region = 'Europe'
GROUP BY c.customer_name, c.region
HAVING max(o.order_date) IS NULL 
    OR max(o.order_date) < '2025-07-01';

customer_name,region,last_order
Cyberdyne Sys,Europe,2025-06-08


## 1.6 : Three Ingredients of a Well-Scoped Dashboard

### Concepts
Every successful dashboard has three ingredients:

1. **Purpose**: What decision does this dashboard support?
   * A dashboard without a purpose is just a data dump.
   * Example: "Help regional managers track quarterly sales and identify at-risk regions."

2. **Audience**: Who consumes it and what do they need?
   * Executives need high-level KPIs. Analysts need drill-down capability.
   * Match the level of detail and interactivity to the audience.

3. **Data**: What data is needed, and is it clean and governed?
   * Pre-join and pre-aggregate in the dataset.
   * Use UC-governed tables for permissions and lineage.
   * Filter to the relevant subset (e.g., closed_won only for revenue).

### Do's and Don'ts

| Do | Don't |
|---|---|
| One clear purpose per dashboard | 20+ widgets covering unrelated topics |
| Pre-aggregate in the dataset | Raw transaction-level data |
| Filter to relevant data (closed_won) | Show all statuses including pending/lost |
| 5-10 visualizations per page | 30+ charts on one page |
| Use UC-governed tables | Ad-hoc CSV uploads with no governance |

In [0]:
%sql
-- 1.6 Demo: Three ingredients applied to our sales dashboard
-- PURPOSE: Track quarterly revenue performance and identify at-risk regions
-- AUDIENCE: Regional sales managers (need numbers, not raw data)
-- DATA: Closed-won orders, by quarter and region, with customer tier

SELECT 
  order_quarter,
  region,
  customer_tier,
  sum(total_amount) AS revenue,
  count(*) AS order_count
FROM module5a_demo1.sales.dashboard_dataset
WHERE status = 'closed_won'
GROUP BY 1, 2, 3
ORDER BY 1, 2, 3;

-- Best practices checklist:
-- ✓ One clear purpose: sales performance tracking
-- ✓ Audience-appropriate: pre-aggregated, no SQL needed
-- ✓ Governed data: UC table, filtered to closed_won
-- ✓ 5-10 visualizations: one dataset powers multiple charts
-- ✓ Scoped: covers sales only, not marketing or operations

order_quarter,region,customer_tier,revenue,order_count
2025-Q1,Asia Pacific,enterprise,2998.00,1
2025-Q1,Europe,premium,3494.00,2
2025-Q1,North America,enterprise,4497.00,1
2025-Q1,North America,standard,598.00,1
2025-Q2,Asia Pacific,premium,2998.00,1
2025-Q2,Europe,enterprise,1495.00,1
2025-Q2,Europe,premium,1497.00,1
2025-Q2,North America,enterprise,1996.00,1
2025-Q2,North America,standard,597.00,1
2025-Q3,Asia Pacific,enterprise,999.00,1


## Genie Space Prompt

### Instructions for the Genie Space

Copy this prompt into the Genie Space instructions when setting up a Genie space for the `module5a_demo1.sales` schema.

---

**Genie Space Instructions:**

You are a sales analytics assistant for a B2B software company. You help users answer questions about sales orders, customers, and products.

**Available tables:**
- `module5a_demo1.sales.orders` — Sales orders with order_date, quantity, unit_price, total_amount, status, channel
- `module5a_demo1.sales.customers` — Customer info with customer_name, customer_tier (enterprise/premium/standard), region
- `module5a_demo1.sales.products` — Product catalog with product_name, category, unit_price
- `module5a_demo1.sales.dashboard_dataset` — Pre-joined view of orders + customers + products (use this for most queries)

**Key definitions:**
- "Revenue" = SUM(total_amount) where status = 'closed_won'
- "Order count" = COUNT(*) where status = 'closed_won'
- "Average order value" = AVG(total_amount) where status = 'closed_won'
- Quarters are derived from order_date using concat(year(order_date), '-Q', quarter(order_date))
- Regions: North America, Europe, Asia Pacific

**Example questions:**
1. "What was our total revenue in Q1?"
2. "Show revenue by region for each quarter"
3. "Which product category generates the most revenue?"
4. "Who are our top 5 customers by total revenue?"
5. "What's the average order value for enterprise customers?"

**Guidelines:**
- Always filter to status = 'closed_won' for revenue calculations
- Use the dashboard_dataset table for most queries (it has all joins pre-built)
- Format dates as yyyy-MM for months or use concat(year(date), '-Q', quarter(date)) for quarters
- Round currency to 2 decimal places
- Limit result sets to 100 rows unless the user asks for more

## Learning Conclusion

### What we demonstrated

| Topic | What was demoed | Key Takeaway |
|---|---|---|
| 1.1 | AI/BI family overview | Same data, four access patterns: Dashboard, Genie, Genie Code, Genie One |
| 1.2 | Designing a dashboard dataset | Pre-join and pre-aggregate so the dashboard doesn't recompute on load |
| 1.3 | Dashboard dataset sources | UC tables (direct), reusable aggregations (shared KPIs), dashboard-local views (one-off) |
| 1.4 | Genie Code | AI generates SQL, picks visualizations, plans full dashboards from NL prompts |
| 1.5 | Dashboard vs. Genie Agent | Dashboard = known/repeated questions; Genie = unknown/ad-hoc questions |
| 1.6 | Well-scoped dashboard | Three ingredients: purpose, audience, data |

### Key principles
* **Dashboards are for known questions**: Build them when the question is repeated and the answer should be visual.
* **Genie is for ad-hoc questions**: Use it when the question is new, exploratory, or one-time.
* **Genie Code accelerates authoring**: It generates SQL, visualizations, and full dashboard plans from natural language.
* **Dataset design matters**: Pre-join and pre-aggregate so dashboards load fast.
* **Scope your dashboard**: One purpose, one audience, clean governed data.

In [0]:
%sql
-- CLEANUP: Decommission everything created in this demo
DROP TABLE IF EXISTS module5a_demo1.sales.dashboard_dataset;
DROP TABLE IF EXISTS module5a_demo1.sales.quarterly_revenue;
DROP TABLE IF EXISTS module5a_demo1.sales.orders;
DROP TABLE IF EXISTS module5a_demo1.sales.customers;
DROP TABLE IF EXISTS module5a_demo1.sales.products;
DROP SCHEMA IF EXISTS module5a_demo1.sales CASCADE;
DROP CATALOG IF EXISTS module5a_demo1 CASCADE;

SHOW CATALOGS LIKE 'module5a_demo1';